In [22]:
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from utils import custom_score
from pathlib import Path

In [25]:
data_dir = Path('../../data')
X = pd.read_csv(data_dir / 'x_train.txt', sep=' ')
y = pd.read_csv(data_dir / 'y_train.txt', sep=' ').values.ravel()

feature_dir = Path("../feature_selection/selected_features.txt")
with open(feature_dir, "r") as f:
    content = f.read()
selected_features = [x.strip().strip("'").strip('"') for x in content.split(",")]

X = X[selected_features]

In [ ]:
data_dir = Path('../../data')
X = pd.read_csv(data_dir / 'x_train.txt', sep=' ')
y = pd.read_csv(data_dir / 'y_train.txt', sep=' ').values.ravel()

feature_dir = Path("../feature_selection/selected_features.txt")
with open(feature_dir, "r") as f:
    content = f.read()
selected_features = [x.strip().strip("'").strip('"') for x in content.split(",")]

X = X[selected_features]
# ── Helper: evaluate one (feature subset, hyperparams) config via CV ──────────
def evaluate_config(X, y, features, params, k=5, max_contacts=1000):
    """
    Returns mean project score across k folds.
    features: which columns of X to use
    params: dict of random forest hyperparameters
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    scores = []
    n_var = len(features)

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[features].iloc[train_idx], X[features].iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)

        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]
        # tak z ciekawosci
        print(f"{accuracy_score(y_val, probs>1/3)}")

        # Decision rule: target customers above threshold, cap at proportional max_contacts
        # (scaled to fold size, since each fold is ~20% of training data)
        fold_cap = int(max_contacts * len(val_idx) / len(y))
        threshold = 1/3  # expected-value break-even: 10p - 5(1-p) > 0
        
        # Take customers with p > threshold, then cap at fold_cap
        candidates = np.where(probs > threshold)[0]
        if len(candidates) > fold_cap:
            top = candidates[np.argsort(-probs[candidates])[:fold_cap]]
        else:
            top = candidates

        y_pred = np.zeros_like(y_val)
        y_pred[top] = 1
        scores.append(custom_score(y_val, y_pred, n_var))

    return np.mean(scores)

# ── Hyperparameter grid  ──────────────────────────────────
param_grid = [
    {"n_estimators": 200, "n_jobs": -1},
    {"n_estimators": 500, "n_jobs": -1},
    {"n_estimators": 700, "n_jobs": -1},
]

# ── Forward selection driven by project score ─────────────────────────────────
n_features = X.shape[1]
selected = ['V11', 'V176', 'V191', 'V255', 'V309']
remaining = list(set(selected_features) - set(selected))
best_score = -np.inf
best_params = None
history = []

while remaining:
    fold_best_score = -np.inf
    fold_best_feat = None
    fold_best_params = None

    for feat in remaining:
        trial_subset = selected + [feat]
        for params in param_grid:
            score = evaluate_config(X, y, trial_subset, params)
            if score > fold_best_score:
                fold_best_score = score
                fold_best_feat = feat
                fold_best_params = params

    # Stop if adding any feature hurts (the −200 penalty isn't paid off)
    if fold_best_score <= best_score:
        print(f"\nNo feature improves score. Stopping.")
        break

    selected.append(fold_best_feat)
    remaining.remove(fold_best_feat)
    best_score = fold_best_score
    best_params = fold_best_params
    history.append((len(selected), fold_best_feat, fold_best_score, fold_best_params))
    print(f"Step {len(selected)}: added feat {fold_best_feat} | "
          f"score = {fold_best_score} | params = {fold_best_params}")

print(f"\n── Final ──")
print(f"Selected {len(selected)} features: {selected}")
print(f"Best params: {best_params}")
print(f"CV score:    {best_score}")

0.549
0.56
0.566
0.546
0.565
0.553
0.553
0.56
0.534
0.555
0.557
0.552
0.563
0.544
0.552
0.54
0.534
0.55
0.559
0.561
0.537
0.531
0.543
0.56
0.559
0.538
0.538
0.549
0.557
0.559
0.53
0.537
0.556
0.544
0.553
0.535
0.545
0.552
0.531
0.553
0.539
0.542
0.55
0.531
0.558
0.552
0.543
0.542
0.547
0.569
0.548
0.535
0.546
0.548
0.564
0.549
0.533
0.551
0.542
0.56
0.558
0.552
0.567
0.562
0.586
0.557
0.546
0.555
0.555
0.579
0.555
0.546
0.556
0.552
0.58
0.554
0.531
0.555
0.538
0.58
0.554
0.532
0.555
0.538
0.568
0.549
0.536
0.56
0.538
0.569
0.55
0.556
0.543
0.541
0.565
0.549
0.543
0.549
0.533
0.563
0.551
0.538
0.549
0.539
0.555
0.54
0.543
0.554
0.519
0.552
0.536
0.538
0.553
0.529
0.553
0.542
0.535
0.555
0.527
0.553
0.55
0.53
0.544
0.536
0.57
0.545
0.538
0.553
0.54
0.571
0.553
0.531
0.549
0.539
0.57
0.536
0.559
0.546
0.551
0.557
0.542
0.553
0.542
0.546
0.551
0.536
0.551
0.55
0.544
0.554
0.538
0.532
0.55
0.54
0.529
0.535
0.537
0.543
0.534
0.539
0.536
0.543
0.55
0.539
0.546
0.546
0.552
0.567
0.534
0.554
0.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

data_dir = Path('../../data')
X = pd.read_csv(data_dir / 'x_train.txt', sep=' ')
y = pd.read_csv(data_dir / 'y_train.txt', sep=' ').values.ravel()

feature_dir = Path("../feature_selection/selected_features.txt")
with open(feature_dir, "r") as f:
    content = f.read()

selected_features = [x.strip().strip("'").strip('"') for x in content.split(",")]
X = X[selected_features]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, objective='binary:logistic')
xgb.fit(X_train, y_train)
pred = xgb.predict_proba(X_test)
print(accuracy_score(y_test, pred[:,1]>1/3))
print(custom_score(y_test, pred[:,1]>1/3, len(X_train.columns)))

0.586
-3090


In [19]:
# do xgb trzeba ustawic wiecej seedow
SEED = 3
# Python randomness
random.seed(SEED)
# NumPy randomness
np.random.seed(SEED)
keep = ['V11', 'V176', 'V191', 'V255', 'V309']
X = X[keep]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, objective='binary:logistic',random_state=SEED, seed=SEED)
xgb.fit(X_train, y_train)
pred = xgb.predict_proba(X_test)
print(accuracy_score(y_test, pred[:,1]>1/3))
print(custom_score(y_test, pred[:,1]>1/3, X_train.shape[1]))

0.57
1240


In [20]:
# do xgb trzeba ustawic wiecej seedow
SEED = 3
# Python randomness
random.seed(SEED)
# NumPy randomness
np.random.seed(SEED)
keep = ['V11', 'V176', 'V191', 'V255', 'V309']
X = X[keep]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

xgb = XGBClassifier(n_estimators=500, learning_rate=0.1, objective='binary:logistic',random_state=SEED, seed=SEED)
xgb.fit(X_train, y_train)
pred = xgb.predict_proba(X_test)
print(accuracy_score(y_test, pred[:,1]>1/3))
print(custom_score(y_test, pred[:,1]>1/3, X_train.shape[1]))

0.57
1240
